In [1]:

import pandas as pd
from glob import glob
import joblib
import json
import time
from copy import deepcopy
import os
import re
import numpy as np
import ee
import ast

In [ ]:
# from google.colab import drive
# drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
agroclimaticZone_acronym_dict = {'Eastern Plateau & Hills Region': 'EPAHR',
                               'Southern Plateau and Hills Region': 'SPAHR',
                               'East Coast Plains & Hills Region': 'ECPHR',
                               'Western Plateau and Hills Region': 'WPAHR',
                               'Central Plateau & Hills Region': 'CPAHR',
                               'Lower Gangetic Plain Region': 'LGPR',
                                'Middle Gangetic Plain Region': 'MGPR',
                                'Eastern Himalayan Region': 'EHR',
                                'Western Himalayan Region': 'WHR',
                                'Upper Gangetic Plain Region': 'UGPR',
                                'Trans Gangetic Plain Region': 'TGPR',
                                'West Coast Plains & Ghat Region': 'WCPGR',
                                'Gujarat Plains & Hills Region': 'GPHR',
                                'Western Dry Region': 'WDR'}

In [4]:
best_month_dict = {'Eastern Plateau & Hills Region': 'cc_12',
                   'Middle Gangetic Plain Region': 'cc_10',
                   'Lower Gangetic Plain Region': 'cc_9',
                   'Western Himalayan Region': 'cc_8',
                   'Eastern Himalayan Region': 'cc_10',
                   'Upper Gangetic Plain Region': 'cc_9',
                   'Trans Gangetic Plain Region': 'cc_9',
                   'Central Plateau & Hills Region': 'cc_7',
                   'Western Plateau and Hills Region': 'cc_11',
                   'Southern Plateau and Hills Region': 'cc_8',
                   'East Coast Plains & Hills Region': 'cc_12'}


In [5]:
# agroclimatic_zone = 'Eastern Plateau & Hills Region'
agroclimatic_zone = 'Southern Plateau and Hills Region'
# agroclimatic_zone = "East Coast Plains & Hills Region"
# agroclimatic_zone = 'Western Plateau and Hills Region'
# agroclimatic_zone = "Central Plateau & Hills Region"
# agroclimatic_zone = 'Lower Gangetic Plain Region'
# agroclimatic_zone = 'Middle Gangetic Plain Region'
# agroclimatic_zone = 'Eastern Himalayan Region'
# agroclimatic_zone = 'Western Himalayan Region'
# agroclimatic_zone = 'Trans Gangetic Plain Region'
# agroclimatic_zone = "Upper Gangetic Plain Region"
# agroclimatic_zone = "West Coast Plains & Ghat Region" # Model not available
# agroclimatic_zone = 'Gujarat Plains & Hills Region' # Model not available
# agroclimatic_zone = 'Western Dry Region' # Model not available

In [6]:
# Set the list of years for which to rename folders
years = ['2017']# ['2016', '2017', '2018','2019','2020','2021','2022','2023','2024']

# CHM

In [8]:
df = pd.read_csv(f'/content/drive/MyDrive/TreeHealth/Agroclimatic_regions/{agroclimatic_zone}.csv')
dist_list = list(df['Name'])
dist_list = ['Anantapur']
print(dist_list)

['Anantapur']


In [9]:
# Set the root directory to the specified folder
root_dir = '/content/drive/MyDrive/'
os.chdir(root_dir)

In [10]:
! pwd

/content/drive/MyDrive


In [11]:
# Get the current directory
current_directory = os.getcwd()

# List all folders in the current directory
folders = [f for f in os.listdir(current_directory) if os.path.isdir(os.path.join(current_directory, f))]

# Print the list of folders
# print("Folders in the current directory:")

for folder in folders:
    print(folder)

core-stack-docs
kapil_test
Colab Notebooks
Pan_india_maps_screenshots
ATCF_ch_5jun25
Google Earth
Datasets
WBC_data
Apache
tiff_export
Pan_india
Aquifer_vector
Aquifer_export
TreeHealth
MGPR
pennar_lulc_23_24
LGPR1
lcw
Untitled form (File responses)
change_test
ET_downloads
core_stack_manual
WHR_2024
WHR_2023
WHR_2022
WHR_2021
WHR_2020
WHR_2019
WHR_2018
WHR_2017
WHR_2016
EPAHR_2016
EPAHR_2017
EPAHR_2018
EPAHR_2019
EPAHR_2020
EPAHR_2021
EPAHR_2022
EPAHR_2023
SPAHR_2016
SPAHR_2017
SPAHR_2018
SPAHR_2019
SPAHR_2020
SPAHR_2021
SPAHR_2022
SPAHR_2023
ECPHR_2016
ECPHR_2017
ECPHR_2018
ECPHR_2019
ECPHR_2020
ECPHR_2021
ECPHR_2022
ECPHR_2023
WPAHR_2016
WPAHR_2017
WPAHR_2018
WPAHR_2019
WPAHR_2020
WPAHR_2021
WPAHR_2022
WPAHR_2023
CPAHR_2016
CPAHR_2017
CPAHR_2018
CPAHR_2019
CPAHR_2020
CPAHR_2021
CPAHR_2022
CPAHR_2023
LGPR_2016
MGPR_2016
EHR_2016
EHR_2017
EHR_2018
EHR_2019
EHR_2020
EHR_2021
EHR_2022
EHR_2023
UGPR_2016
UGPR_2017
UGPR_2018
UGPR_2019
UGPR_2020
UGPR_2021
UGPR_2022
UGPR_2023
TGPR_2016
TGPR

In [12]:
# Regular expression pattern to match folder names starting with a particular pattern
pattern = re.compile(r'^' + f'{agroclimaticZone_acronym_dict[agroclimatic_zone]}')

# Filter folders based on the pattern
matching_folders = [folder for folder in folders if pattern.match(folder)]

# Print the matching folders
print(f"Folders starting with '{agroclimaticZone_acronym_dict[agroclimatic_zone]}':")
for folder in matching_folders:
    print(folder)

Folders starting with 'SPAHR':
SPAHR_2016
SPAHR_2017
SPAHR_2018
SPAHR_2019
SPAHR_2020
SPAHR_2021
SPAHR_2022
SPAHR_2023
SPAHR_2024


In [13]:
# Leh (Ladkh) is the only district encountered having special character '(' and ')' in it. That's why its handled in a special way as seen here

# Transfer files from duplicate folders to original folder

dist_num = 0
# for district in dist_list:
#     print(dist_num)
for year in years:
    print(year)

    # orig_district = district
    # if district == 'Leh (Ladakh)':
    #     district = 'Leh'

    # pattern = re.compile(r'^' + agroclimaticZone_acronym_dict[agroclimatic_zone] + '_' + district + '_' + year)
    pattern = re.compile(r'^' + agroclimaticZone_acronym_dict[agroclimatic_zone] + '_' + year)
    district_year_folders = [folder for folder in matching_folders if pattern.match(folder)]
    print("district_year_folders-->", district_year_folders)

    # district = orig_district

    while len(district_year_folders) > 1:
        source_folder = district_year_folders[0]
        destination_folder = district_year_folders[1]

        files_to_move = os.listdir(source_folder)
        for file_name in files_to_move:
            source_path = os.path.join(source_folder, file_name)
            destination_path = os.path.join(destination_folder, file_name)
            os.rename(source_path, destination_path)

        del district_year_folders[0]
    if len(district_year_folders) > 0:
      current_folder_name = district_year_folders[0]
      # new_folder_name = f'{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{district}_{year}'
      new_folder_name = f'{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{year}'
      os.rename(current_folder_name, new_folder_name)

    # dist_num += 1

2017
district_year_folders--> ['SPAHR_2017']


In [14]:
# Check all folders with more than 0 files
dist_num = 0
# for district in dist_list:
# print(dist_num)
for year in years:
    # orig_district = district
    # if district == 'Leh (Ladakh)':
    #     district = 'Leh'

    # Re-scan the current directory for folders after renaming
    current_directory = os.getcwd()
    folders = [f for f in os.listdir(current_directory) if os.path.isdir(os.path.join(current_directory, f))]
    # pattern = re.compile(r'^' + agroclimaticZone_acronym_dict[agroclimatic_zone] + '_' + district + '_' + year)
    pattern = re.compile(r'^' + agroclimaticZone_acronym_dict[agroclimatic_zone] + '_' + year)
    district_year_folders = [folder for folder in folders if pattern.match(folder)]

    # district = orig_district

    for folder in district_year_folders:
        if len(os.listdir(folder)) > 0:
            print(folder)

    # dist_num += 1

SPAHR_2017


In [15]:
# Change back to parent directory
os.chdir(os.path.dirname(os.getcwd()))

In [16]:
! pwd

/content/drive


In [17]:
df = pd.read_csv('/content/drive/MyDrive/TreeHealth/district_to_agroclimaticZone_mapping.csv')
print(df.shape[0])

666


In [18]:
# Function to convert string representation of list to an actual list
def convert_to_list(string):
    return ast.literal_eval(string)

df['IntersectingZones'] = df['IntersectingZones'].apply(convert_to_list)
print(df)

                     District  \
0             Nicobar Islands   
1    North and Middle Andaman   
2               South Andaman   
3                   Anantapur   
4                    Chittoor   
..                        ...   
661        Pashchim Medinipur   
662           Purba Medinipur   
663                  Puruliya   
664         South 24 Parganas   
665            Uttar Dinajpur   

                                     IntersectingZones  \
0                                                   []   
1                                                   []   
2                                                   []   
3                  [Southern Plateau and Hills Region]   
4    [Southern Plateau and Hills Region, East Coast...   
..                                                 ...   
661  [East Coast Plains & Hills Region, Eastern Pla...   
662  [East Coast Plains & Hills Region, Lower Gange...   
663  [Eastern Plateau & Hills Region, Lower Gangeti...   
664                    

In [19]:
district_mapping_df = df[df['AgroclimaticZone'] == agroclimatic_zone][['District', 'IntersectingZones']]
district_mapping_df= district_mapping_df[district_mapping_df['District'].isin(dist_list)] ## Added extra
print(district_mapping_df)

    District                    IntersectingZones
3  Anantapur  [Southern Plateau and Hills Region]


In [20]:

# print(district_mapping_df['IntersectingZones'][165])
# district_mapping_df['IntersectingZones']
i = 0
for ind in district_mapping_df.index:
    district = district_mapping_df.loc[ind, 'District']
    zones = district_mapping_df['IntersectingZones'][ind]
    print(i, district, zones)
    i += 1
    # for zone in zones:
    #     if zone not in agroclimaticZone_acronym_dict:
    #         print(district, zone)

0 Anantapur ['Southern Plateau and Hills Region']


In [21]:
agroclimatic_zone_model_path_mapping_rh98 = {'Eastern Plateau & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Eastern_Plateau_Hills_Region_correct_toa_rh98_24.joblib',
                                             'East Coast Plains & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/East_Coast_Plains_Hills_Region_correct_toa_rh98_23.joblib',
                                             'Western Himalayan Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Western_Himalayan_Region_correct_toa_rh98_30.joblib',
                                             'Eastern Himalayan Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Eastern_Himalayan_Region_correct_toa_rh98_25.joblib',
                                             'Central Plateau & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Central_Plateau_Hills_Region_correct_toa_rh98_23.joblib',
                                             'Southern Plateau and Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Southern_Plateau_and_Hills_Region_correct_toa_rh98_23.joblib',
                                             'Western Plateau and Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Western_Plateau_and_Hills_Region_correct_toa_rh98_24.joblib',
                                             'Upper Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Upper_Gangetic_Plain_Region_correct_toa_rh98_29.joblib',
                                             'Middle Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Middle_Gangetic_Plain_Region_correct_toa_rh98_24.joblib',
                                             'Trans Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Trans_Gangetic_Plain_Region_correct_toa_rh98_21.joblib',
                                             'Lower Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Lower_Gangetic_Plain_Region_correct_toa_rh98_17.joblib'}

agroclimatic_zone_model_path_mapping_rh75 = {'Eastern Plateau & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Eastern_Plateau_Hills_Region_correct_toa_rh75_17.joblib',
                                             'East Coast Plains & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/East_Coast_Plains_Hills_Region_correct_toa_rh75_16.joblib',
                                             'Western Himalayan Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Western_Himalayan_Region_correct_toa_rh75_20.joblib',
                                             'Eastern Himalayan Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Eastern_Himalayan_Region_correct_toa_rh75_18.joblib',
                                             'Central Plateau & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Central_Plateau_Hills_Region_correct_toa_rh75_16.joblib',
                                             'Southern Plateau and Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Southern_Plateau_and_Hills_Region_correct_toa_rh75_16.joblib',
                                             'Western Plateau and Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Western_Plateau_and_Hills_Region_correct_toa_rh75_17.joblib',
                                             'Upper Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Upper_Gangetic_Plain_Region_correct_toa_rh75_22.joblib',
                                             'Middle Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Middle_Gangetic_Plain_Region_correct_toa_rh75_17.joblib',
                                             'Trans Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Trans_Gangetic_Plain_Region_correct_toa_rh75_15.joblib',
                                             'Lower Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Lower_Gangetic_Plain_Region_correct_toa_rh75_12.joblib'}

agroclimatic_zone_model_path_mapping_rh50 = {'Eastern Plateau & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Eastern_Plateau_Hills_Region_correct_toa_rh50_12.joblib',
                                             'East Coast Plains & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/East_Coast_Plains_Hills_Region_correct_toa_rh50_12.joblib',
                                             'Western Himalayan Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Western_Himalayan_Region_correct_toa_rh50_14.joblib',
                                             'Eastern Himalayan Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Eastern_Himalayan_Region_correct_toa_rh50_13.joblib',
                                             'Central Plateau & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Central_Plateau_Hills_Region_correct_toa_rh50_11.joblib',
                                             'Southern Plateau and Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Southern_Plateau_and_Hills_Region_correct_toa_rh50_12.joblib',
                                             'Western Plateau and Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Western_Plateau_and_Hills_Region_correct_toa_rh50_12.joblib',
                                             'Upper Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Upper_Gangetic_Plain_Region_correct_toa_rh50_17.joblib',
                                             'Middle Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Middle_Gangetic_Plain_Region_correct_toa_rh50_12.joblib',
                                             'Trans Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Trans_Gangetic_Plain_Region_correct_toa_rh50_11.joblib',
                                             'Lower Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/chm_final/Lower_Gangetic_Plain_Region_correct_toa_rh50_9.joblib'}

In [22]:
MODEL_PATH_rh98 = agroclimatic_zone_model_path_mapping_rh98[agroclimatic_zone]
model_rh98 = joblib.load(MODEL_PATH_rh98)

MODEL_PATH_rh75 = agroclimatic_zone_model_path_mapping_rh75[agroclimatic_zone]
model_rh75 = joblib.load(MODEL_PATH_rh75)

MODEL_PATH_rh50 = agroclimatic_zone_model_path_mapping_rh50[agroclimatic_zone]
model_rh50 = joblib.load(MODEL_PATH_rh50)

/usr/lib/python3.12/pickle.py:1760: UserWarning: [15:20:26] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [23]:
if hasattr(model_rh98, 'feature_names_in_'):
    features_rh98 = model_rh98.feature_names_in_

if hasattr(model_rh75, 'feature_names_in_'):
    features_rh75 = model_rh75.feature_names_in_

if hasattr(model_rh50, 'feature_names_in_'):
    features_rh50 = model_rh50.feature_names_in_


In [24]:
seasons = ['kharif', 'rabi', 'zaid']

In [25]:
def add_s1_indices(df):
    for season in seasons:
        # SAR Indices
        df[f'VV_VH_Ratio_{season}'] = df[f'VV_{season}'] / df[f'VH_{season}']
        df[f'VH_VV_Ratio_{season}'] = df[f'VH_{season}'] / df[f'VV_{season}']
        df[f'SAR_NDVI_{season}'] = (df[f'VH_{season}'] - df[f'VV_{season}']) / (df[f'VH_{season}'] + df[f'VV_{season}'])
        df[f'SAR_DVI_{season}'] = df[f'VH_{season}'] - df[f'VV_{season}']
        df[f'SAR_SVI_{season}'] = df[f'VH_{season}'] + df[f'VV_{season}']
        df[f'SAR_RDVI_{season}'] = (df[f'VH_{season}'] / df[f'VV_{season}']) - (df[f'VV_{season}'] / df[f'VH_{season}'])
        df[f'SAR_NRDVI_{season}'] = ((df[f'VH_{season}']/df[f'VV_{season}'] - df[f'VV_{season}']/df[f'VH_{season}']) / (df[f'VH_{season}']/df[f'VV_{season}'] + df[f'VV_{season}']/df[f'VH_{season}']))
        df[f'SAR_SSDVI_{season}'] = df[f'VH_{season}']**2 - df[f'VV_{season}']**2

def add_s2_indices(df):
    for season in seasons:
        # Optical Indices
        df[f'NDVI_{season}'] = (df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + df[f'B4_{season}'])
        df[f'NDWI_{season}'] = (df[f'B8_{season}'] - df[f'B12_{season}']) / (df[f'B8_{season}'] + df[f'B12_{season}'])
        df[f'EVI_{season}'] = (2.5 * ((df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + 6*df[f'B4_{season}'] - 7.5*df[f'B2_{season}'] + 1)))
        df[f'OSAVI_{season}'] = (df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + df[f'B4_{season}'] + 0.16)
        df[f'ARVI_{season}'] = (df[f'B8_{season}'] - 2*df[f'B4_{season}'] + df[f'B2_{season}']) / (df[f'B8_{season}'] + 2*df[f'B4_{season}'] + df[f'B2_{season}'])
        df[f'VARI_{season}'] = (df[f'B3_{season}'] - df[f'B4_{season}']) / (df[f'B3_{season}'] + df[f'B4_{season}'] - df[f'B2_{season}'])


In [26]:
# def get_csv_data(fileName):
#     data = pd.DataFrame()
#     try:
#         data = pd.read_csv(fileName)
#     except Exception as exp:
#         print("Error reading file ", fileName, " - ", exp)
#     return data

def get_csv_chunks(fileName, chunksize=100_000):
    """Yield DataFrame chunks from a CSV file."""
    try:
        for chunk in pd.read_csv(fileName, chunksize=chunksize):
            yield chunk
    except Exception as exp:
        print("Error reading file ", fileName, " - ", exp)
        return []


In [27]:
# For Canopy Height
def pipeline(fileName, chunksize=100000):
    res_chunks = []  # collect processed results

    for df in get_csv_chunks(fileName, chunksize):
        if len(df) == 0:
            continue

        # Add indices
        add_s1_indices(df)
        add_s2_indices(df)

        # Geo column
        res_df = pd.DataFrame()
        res_df['.geo'] = df['.geo']

        # Predictions
        pred_y_98 = model_rh98.predict(df[features_rh98])
        pred_y_75 = model_rh75.predict(df[features_rh75])
        pred_y_50 = model_rh50.predict(df[features_rh50])

        res_df['rh98_class'] = pred_y_98
        res_df['rh75_class'] = pred_y_75
        res_df['rh50_class'] = pred_y_50

        res_chunks.append(res_df)

    if res_chunks:
        return pd.concat(res_chunks, ignore_index=True)
    else:
        return pd.DataFrame(columns=['.geo', 'rh98_class', 'rh75_class', 'rh50_class'])

# def pipeline(fileName):
#     # print(fileName)

#     df = get_csv_data(fileName)

#     if (len(df) == 0):
#         return df

#     add_s1_indices(df)
#     add_s2_indices(df)

#     geoList = list(df['.geo'])
#     res_df = pd.DataFrame()
#     res_df['.geo'] = geoList

#     test_df = df[features_rh98]
#     pred_y_98 = list(model_rh98.predict(test_df))
#     test_df = df[features_rh75]
#     pred_y_75 = list(model_rh75.predict(test_df))
#     test_df = df[features_rh50]
#     pred_y_50 = list(model_rh50.predict(test_df))

#     res_df['rh98_class'] = pred_y_98
#     res_df['rh75_class'] = pred_y_75
#     res_df['rh50_class'] = pred_y_50

#     return res_df

In [30]:
for year in years:
    dist_num = 0

    for ind in district_mapping_df.index:
        # if dist_num < 14:
        #     dist_num += 1
        #     continue
        district = district_mapping_df.loc[ind, 'District']
        print(district)
        zones = district_mapping_df['IntersectingZones'][ind]
        print(zones)
        merged_df = pd.DataFrame()
        for zone in zones:
            print(f'\n{dist_num} District: {district}, Zone: {zone}, Year: {year}')
            # dist_data_path = f'/content/drive/MyDrive/{agroclimaticZone_acronym_dict[zone]}_{district}_{year}/'
            dist_data_path = f'/content/drive/MyDrive/{agroclimaticZone_acronym_dict[zone]}_{year}'
            files = glob(f"{dist_data_path}/{district}_{year}_all_grids.csv")
            print("no. of files:", len(files), '\n')
            for fileName in files:
                df = pipeline(fileName,chunksize=100000)
                merged_df = pd.concat([merged_df, df], ignore_index=True)

        merged_df.to_csv(f'/content/drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{year}/result_chm.csv', index=False)
        dist_num += 1

Anantapur
['Southern Plateau and Hills Region']

0 District: Anantapur, Zone: Southern Plateau and Hills Region, Year: 2017
no. of files: 1 



# CCD

In [ ]:
df = pd.read_csv(f'/content/drive/MyDrive/TreeHealth/Agroclimatic_regions/{agroclimatic_zone}.csv')
dist_list =  list(df['Name'])
# dist_list = ['Yamunanagar'] ## Added extra

In [ ]:
print(len(dist_list))
print(dist_list)

86
['Araria', 'Arwal', 'AurangabadB', 'Banka', 'Begusarai', 'Bhagalpur', 'Bhojpur', 'Buxar', 'Darbhanga', 'Gaya', 'Gopalganj', 'Jamui', 'Jehanabad', 'Kaimur', 'Katihar', 'Khagaria', 'Kishanganj', 'Lakhisarai', 'Madhepura', 'Madhubani', 'Munger', 'Muzaffarpur', 'Nalanda', 'Nawada', 'Pashchim Champaran', 'Patna', 'Purba Champaran', 'Purnia', 'Rohtas', 'Saharsa', 'Samastipur', 'Saran', 'Sheikhpura', 'Sheohar', 'Sitamarhi', 'Siwan', 'Supaul', 'Vaishali', 'BalrampurC', 'Chatra', 'Deoghar', 'Dumka', 'Garhwa', 'Giridih', 'Godda', 'Hazaribagh', 'Kodarma', 'Pakur', 'Palamu', 'Sahibganj', 'Rewa', 'Singrauli', 'Allahabad', 'Ambedkar Nagar', 'Amethi', 'Azamgarh', 'Bahraich', 'Ballia', 'Balrampur', 'Barabanki', 'Basti', 'Chandauli', 'Deoria', 'Faizabad', 'Ghazipur', 'Gonda', 'Gorakhpur', 'Jaunpur', 'Kushinagar', 'Lakhimpur Kheri', 'Maharajganj', 'Mau', 'Mirzapur', 'Pratapgarhup', 'Sant Kabir Nagar', 'Sant Ravi Das Nagar', 'Shravasti', 'Siddharth Nagar', 'Sitapur', 'Sonbhadra', 'Sultanpur', 'Varanas

In [ ]:
agroclimatic_zone_model_path_mapping = {'Central Plateau & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Central_Plateau_Hills_Region_toa_monthly_cover_51.joblib',
                                        'Lower Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Lower_Gangetic_Plain_Region_toa_monthly_cover_48.joblib',
                                        'Middle Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Middle_Gangetic_Plain_Region_toa_monthly_cover_50.joblib',
                                        'Eastern Himalayan Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Eastern_Himalayan_Region_toa_monthly_cover_86.joblib',
                                        'Western Himalayan Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Western_Himalayan_Region_toa_monthly_cover_78.joblib',
                                        'Upper Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Upper_Gangetic_Plain_Region_toa_monthly_cover_67.joblib',
                                        'Trans Gangetic Plain Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Trans_Gangetic_Plain_Region_toa_monthly_cover_55.joblib',
                                        'East Coast Plains & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/East_Coast_Plains_Hills_Region_toa_monthly_cover_67.joblib',
                                        'Eastern Plateau & Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Eastern_Plateau_Hills_Region_toa_monthly_cover_60.joblib',
                                        'Western Plateau and Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Western_Plateau_and_Hills_Region_toa_monthly_cover_57.joblib',
                                        'Southern Plateau and Hills Region': '/content/drive/MyDrive/TreeHealth/best_models/corrected/Southern_Plateau_and_Hills_Region_toa_monthly_cover_62.joblib'}


In [ ]:
MODEL_PATH_cc = agroclimatic_zone_model_path_mapping[agroclimatic_zone]
model_cc = joblib.load(MODEL_PATH_cc)

/usr/lib/python3.12/pickle.py:1760: UserWarning: [07:18:56] WARNING: /workspace/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [ ]:
if hasattr(model_cc, 'feature_names_in_'):
    features_cc = model_cc.feature_names_in_

In [ ]:
seasons = ['kharif', 'rabi', 'zaid']


In [ ]:
def add_s1_indices(df):
    for season in seasons:
        # SAR Indices
        df[f'VV_VH_Ratio_{season}'] = df[f'VV_{season}'] / df[f'VH_{season}']
        df[f'VH_VV_Ratio_{season}'] = df[f'VH_{season}'] / df[f'VV_{season}']
        df[f'SAR_NDVI_{season}'] = (df[f'VH_{season}'] - df[f'VV_{season}']) / (df[f'VH_{season}'] + df[f'VV_{season}'])
        df[f'SAR_DVI_{season}'] = df[f'VH_{season}'] - df[f'VV_{season}']
        df[f'SAR_SVI_{season}'] = df[f'VH_{season}'] + df[f'VV_{season}']
        df[f'SAR_RDVI_{season}'] = (df[f'VH_{season}'] / df[f'VV_{season}']) - (df[f'VV_{season}'] / df[f'VH_{season}'])
        df[f'SAR_NRDVI_{season}'] = ((df[f'VH_{season}']/df[f'VV_{season}'] - df[f'VV_{season}']/df[f'VH_{season}']) / (df[f'VH_{season}']/df[f'VV_{season}'] + df[f'VV_{season}']/df[f'VH_{season}']))
        df[f'SAR_SSDVI_{season}'] = df[f'VH_{season}']**2 - df[f'VV_{season}']**2

def add_s2_indices(df):
    for season in seasons:
        # Optical Indices
        df[f'NDVI_{season}'] = (df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + df[f'B4_{season}'])
        df[f'NDWI_{season}'] = (df[f'B8_{season}'] - df[f'B12_{season}']) / (df[f'B8_{season}'] + df[f'B12_{season}'])
        df[f'EVI_{season}'] = (2.5 * ((df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + 6*df[f'B4_{season}'] - 7.5*df[f'B2_{season}'] + 1)))
        df[f'OSAVI_{season}'] = (df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + df[f'B4_{season}'] + 0.16)
        df[f'ARVI_{season}'] = (df[f'B8_{season}'] - 2*df[f'B4_{season}'] + df[f'B2_{season}']) / (df[f'B8_{season}'] + 2*df[f'B4_{season}'] + df[f'B2_{season}'])
        df[f'VARI_{season}'] = (df[f'B3_{season}'] - df[f'B4_{season}']) / (df[f'B3_{season}'] + df[f'B4_{season}'] - df[f'B2_{season}'])


In [ ]:
# def get_csv_data(fileName):
#     data = pd.DataFrame()
#     try:
#         data = pd.read_csv(fileName)
#     except Exception as exp:
#         print("Error reading file ", fileName, " - ", exp)
#     return data

def get_csv_chunks(fileName, chunksize=100_000):
    """Yield DataFrame chunks from a CSV file."""
    try:
        for chunk in pd.read_csv(fileName, chunksize=chunksize):
            yield chunk
    except Exception as exp:
        print("Error reading file ", fileName, " - ", exp)
        return []

In [ ]:
# For Canopy Cover
def pipeline(fileName, chunksize=100000):
    res_chunks = []  # collect processed results

    for df in get_csv_chunks(fileName, chunksize):
        if len(df) == 0:
            continue

        # Add indices
        add_s1_indices(df)
        add_s2_indices(df)

        # Geo column
        res_df = pd.DataFrame()
        res_df['.geo'] = list(df['.geo'])

        for month in range(1,13):
            df['month_sin'] = [np.sin(2 * np.pi * month / 12)] * len(df)
            df['month_cos'] = [np.cos(2 * np.pi * month / 12)] * len(df)

            test_df = df[features_cc]
            pred_y_cc = list(model_cc.predict(test_df))
            res_df[f'cc_{month}'] = pred_y_cc

        res_chunks.append(res_df)

    if res_chunks:
        return pd.concat(res_chunks, ignore_index=True)
    else:
        return pd.DataFrame(columns=['.geo'])

# def pipeline(fileName):

#     print(fileName)

#     df = get_csv_data(fileName)

#     if (len(df) == 0):
#         return df

#     add_s1_indices(df)
#     add_s2_indices(df)

#     geoList = list(df['.geo'])
#     res_df = pd.DataFrame()
#     res_df['.geo'] = geoList

#     for month in range(1,13):
#         df['month_sin'] = [np.sin(2 * np.pi * month / 12)] * len(df)
#         df['month_cos'] = [np.cos(2 * np.pi * month / 12)] * len(df)

#         test_df = df[features_cc]
#         pred_y_cc = list(model_cc.predict(test_df))
#         res_df[f'cc_{month}'] = pred_y_cc

#     return res_df


In [ ]:
for year in years:
    dist_num = 0
    for district in dist_list:
        # if dist_num < 53:
        #     dist_num += 1
        #     continue
        print('\n', dist_num, district, year)
        # dist_data_path = f'/content/drive/MyDrive/{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{district}_{year}/'
        dist_data_path = f'/content/drive/MyDrive/{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{year}'
        print(dist_data_path)
        # files = glob(dist_data_path + "*.csv")
        files = glob(f"{dist_data_path}/{district}_{year}_all_grids.csv")
        print("no. of files:", len(files), '\n')
        merged_df = pd.DataFrame()
        for fileName in files:
            df = pipeline(fileName, chunksize=100000)
            merged_df = pd.concat([merged_df, df], ignore_index=True)
        print("merged_df", len(merged_df))
        merged_df.to_csv(f'/content/drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{year}/result_monthly_cc.csv', index=False)
        dist_num += 1


 0 Araria 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 1976372

 1 Arwal 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 192050

 2 AurangabadB 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 931520

 3 Banka 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 1794108

 4 Begusarai 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 925926

 5 Bhagalpur 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 1017500

 6 Bhojpur 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 528965

 7 Buxar 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 319698

 8 Darbhanga 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 1482059

 9 Gaya 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 2347878

 10 Gopalganj 2021
/content/drive/MyDrive/MGPR_2021
no. of files: 1 

merged_df 960264

 11 Jamui 2021
/content/drive/MyDrive/MGPR_2021
no. of f